## Import

In [2]:
%load_ext autoreload
%autoreload 2

In [21]:
import duckdb

from sindex.sources.uspto.discovery import process_patents_zip_parallel
from sindex.sources.uspto.utils import count_patents_in_zip_folder
from sindex.sources.uspto.jobs import extract_uspto_mentions

## Extract IDs (with regex) from patent zip files

### Paths

In [4]:
#input_dir=r"D:\pipeline-data\mentions\uspto\not_needed\test_zip"
#output_dir=r"D:\pipeline-data\mentions\uspto\not_needed\test_output"

In [26]:
input_dir=r"D:\pipeline-data\mentions\uspto\PTGRXML"
output_dir=r"D:\pipeline-data\mentions\uspto\processed_zip_ndjson"

### Run extraction

In [27]:
process_patents_zip_parallel(input_dir, output_dir)

1100 zip files found in D:\pipeline-data\mentions\uspto\PTGRXML
241 matching outputs already exist in D:\pipeline-data\mentions\uspto\processed_zip_ndjson
Processing remaining 859 files
Progress: 859/859 | (Scanned 4682162 patents, found 92723 with relevant IDs)                        
\Done!
New Patents Scanned:      4,682,162
New Patents Saved:        92,723
Total New DOIs Found:     324,978
Total New EMDB IDs Found: 1,433
Output saved to: D:\pipeline-data\mentions\uspto\processed_zip_ndjson


### Stats

In [28]:
count_patents_in_zip_folder(input_dir)

Found 1100 zip files. Starting fast count...
Scanned 1100/1100 files. Total Patents so far: 6,445,063
Done counting!
Total Files Scanned: 1100
Total Patents Found: 6,445,063


In [29]:
count_outputs_stats(output_dir)

Scanning 1100 files using orjson...
Scanned 1100/1100 | Patents: 184,257 | DOIs: 789,558 | EMDBs: 1,858                                 
Done counting!
Files Processed:      1,100
Total Patents Found:  184,257
Total DOIs Found:     789,558
Total EMDB IDs Found: 1,858


## Find mentions to our datasets

### Create db with all dataset IDs (should be done, can skip)

In [17]:
# Use raw strings for Windows paths to avoid escape character issues
output_db = r"D:\pipeline-data\records\slim-records\all-slim-records.duckdb"
datacite_db = r"D:\pipeline-data\records\slim-records\datacite-slim-records.duckdb"
emdb_db = r"D:\pipeline-data\records\slim-records\emdb-slim-records.duckdb"

In [15]:
con = duckdb.connect(output_db)

con.execute(f"ATTACH '{datacite_db}' AS db1")
con.execute(f"ATTACH '{emdb_db}' AS db2")

con.execute("""
    CREATE OR REPLACE TABLE my_datasets_all AS 
    SELECT * FROM db1.my_datasets
    UNION ALL
    SELECT * FROM db2.my_datasets
""")

count = con.execute("SELECT count(*) FROM my_datasets_all").fetchone()[0]
print(f"Merged {count} rows into my_datasets_all")

con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merged 49061167 rows into my_datasets_all


### Compare and create mentions file

In [31]:
dataset_ids_db = r"D:\pipeline-data\records\slim-records\all-slim-records.duckdb"
patents_processed_ndjson_dir = r"D:\pipeline-data\mentions\uspto\processed_zip_ndjson"
output_mentions_ndjson = r"D:\pipeline-data\mentions\uspto\uspto_mentions.ndjson"
extract_uspto_mentions(dataset_ids_db, patents_processed_ndjson_dir, output_mentions_ndjson)

Enriching 1100 USPTO files against D:\pipeline-data\records\slim-records\all-slim-records.duckdb...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done! Matches found: 1,519
Saved to: D:\pipeline-data\mentions\uspto\uspto_mentions.ndjson
